In [ ]:
!pip install torch torchaudio datasets transformers numpy tqdm soundfile huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


###Change Sampling rate to 44.1kHz

In [ ]:
import torch
import torchaudio
import datasets
from datasets import load_dataset, Dataset, Audio
import numpy as np
from tqdm.auto import tqdm
import logging
import gc

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('resampling.log'),
        logging.StreamHandler()
    ]
)

def setup_processing():
    """Setup processing environment and return device"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    logging.info(f"Using device: {device}")
    return device

def resample_audio(audio_data, orig_sample_rate, target_sample_rate=44100, device="cpu"):
    """Resample audio and return numpy array"""
    try:
        # Convert to tensor if needed
        if not isinstance(audio_data, torch.Tensor):
            audio_data = torch.tensor(audio_data)

        # Convert to double precision
        audio_data = audio_data.to(torch.float64)

        # Move to device
        audio_data = audio_data.to(device)

        # Create resampler with explicit dtype
        resampler = torchaudio.transforms.Resample(
            orig_freq=orig_sample_rate,
            new_freq=target_sample_rate,
            dtype=torch.float64
        ).to(device)

        # Resample
        resampled_audio = resampler(audio_data)

        # Move back to CPU, convert to float32
        resampled_audio = resampled_audio.cpu().to(torch.float32).numpy()

        # Ensure audio is within valid range
        resampled_audio = np.clip(resampled_audio, -1.0, 1.0)

        return resampled_audio, target_sample_rate

    except Exception as e:
        logging.error(f"Error during resampling: {str(e)}")
        raise

def process_batch(batch, start_idx, device):
    """Process a batch of audio samples"""
    processed_batch = {
        "audio": [],
        "text": [],
        "speaker_id": [],
        "style": [],
        "id": [],
        "text_description": []
    }

    for idx in range(len(batch['audio'])):
        try:
            # Get original audio data
            audio_data = batch['audio'][idx]['array']
            orig_sr = batch['audio'][idx]['sampling_rate']

            # Resample audio
            resampled_audio, new_sr = resample_audio(
                audio_data,
                orig_sr,
                target_sample_rate=44100,
                device=device
            )

            # Store processed data
            processed_batch["audio"].append({
                "array": resampled_audio,
                "sampling_rate": new_sr
            })
            processed_batch["text"].append(batch["text"][idx])
            processed_batch["speaker_id"].append(batch["speaker_id"][idx])
            processed_batch["style"].append(batch["style"][idx])
            processed_batch["id"].append(batch["id"][idx])
            processed_batch["text_description"].append(batch["text_description"][idx])

            # Log successful processing
            if idx % 10 == 0:  # Log every 10th sample
                logging.info(f"Successfully processed sample {start_idx + idx}")

        except Exception as e:
            logging.error(f"Error processing sample {start_idx + idx}: {str(e)}")
            continue

        # Clear CUDA cache if using GPU
        if device == "cuda":
            torch.cuda.empty_cache()

    return processed_batch

def main():
    try:
        # Load dataset
        dataset = load_dataset("En1gma02/english_emotions_tagged")
        logging.info("Dataset loaded successfully")

        # Setup device
        device = setup_processing()

        # Process in batches
        batch_size = 16  # Reduced batch size to be safer with memory
        num_batches = len(dataset['train']) // batch_size + 1

        processed_datasets = []

        for i in tqdm(range(num_batches)):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, len(dataset['train']))

            # Get batch
            batch = dataset['train'][start_idx:end_idx]

            # Process batch
            processed_batch = process_batch(batch, start_idx, device)

            # Create dataset from batch
            if processed_batch["audio"]:  # Only if batch contains processed samples
                # Convert to Dataset with proper Audio feature
                batch_dataset = Dataset.from_dict(
                    processed_batch,
                    features=datasets.Features({
                        "audio": datasets.Audio(sampling_rate=44100),
                        "text": datasets.Value("string"),
                        "speaker_id": datasets.Value("string"),
                        "style": datasets.Value("string"),
                        "id": datasets.Value("string"),
                        "text_description": datasets.Value("string")
                    })
                )
                processed_datasets.append(batch_dataset)

            # Clear memory
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

            # Log progress
            logging.info(f"Completed batch {i+1}/{num_batches}")

        # Concatenate all processed batches
        final_dataset = datasets.concatenate_datasets(processed_datasets)

        # Push to hub
        final_dataset.push_to_hub(
            "En1gma02/processed_english_emotions_tagged",
            private=False
        )

        logging.info("Processing completed successfully")

    except Exception as e:
        logging.error(f"Fatal error: {str(e)}")
        raise

if __name__ == "__main__":
    main()

  0%|          | 0/182 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/1453 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

Map:   0%|          | 0/1452 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

###Rename Row

In [ ]:
from datasets import load_dataset

# Step 1: Load the dataset from Hugging Face (replace 'your_dataset_name' with the actual dataset name)
dataset = load_dataset("En1gma02/processed_hindi_speech_male_5hr")

# Step 2: Rename the column 'text' to 'text_hindi' in-place
dataset = dataset.rename_column("text", "text_hindi")

# Step 3: Push the updated dataset back to Hugging Face (replace with your dataset repo name)
dataset.push_to_hub("En1gma02/processed_hindi_speech_male_5hr")


Uploading the dataset shards:   0%|          | 0/9 [00:00<?, ?it/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Map:   0%|          | 0/649 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

...

Map:   0%|          | 0/649 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Map:   0%|          | 0/649 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/En1gma02/processed_hindi_speech_male_5hr/commit/d431e8a0607ff0e9cd19e87568b4315a9b39faa2', commit_message='Upload dataset', commit_description='', oid='d431e8a0607ff0e9cd19e87568b4315a9b39faa2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/En1gma02/processed_hindi_speech_male_5hr', endpoint='https://huggingface.co', repo_type='dataset', repo_id='En1gma02/processed_hindi_speech_male_5hr'), pr_revision=None, pr_num=None)

###Check Sampling Rate

In [ ]:
 from datasets import load_dataset

# Load the processed dataset
dataset = load_dataset("En1gma02/processed_english_emotions_tagged")

# Take the first 5 rows and print their sample rates
for i in range(5):
    print(f"Sample {i}:")
    print(f"Sampling Rate: {dataset['train'][i]['audio']['sampling_rate']} Hz")
    print(f"Audio Array Shape: {dataset['train'][i]['audio']['array'].shape}")
    print("---")

README.md:   0%|          | 0.00/506 [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/400M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2905 [00:00<?, ? examples/s]

Sample 0:
Sampling Rate: 44100 Hz
Audio Array Shape: (249564,)
---
Sample 1:
Sampling Rate: 44100 Hz
Audio Array Shape: (115336,)
---
Sample 2:
Sampling Rate: 44100 Hz
Audio Array Shape: (95193,)
---
Sample 3:
Sampling Rate: 44100 Hz
Audio Array Shape: (102993,)
---
Sample 4:
Sampling Rate: 44100 Hz
Audio Array Shape: (110680,)
---


###Clone Dataset

In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi, HfFolder, create_repo
import os

# Step 1: Set up your Hugging Face credentials
hf_token = "YOUR_HF_TOKEN_HERE"  # Replace with your actual token
HfFolder.save_token(hf_token)

# Step 2: Load the dataset you want to clone
source_dataset_name = "ylacombe/expresso"  # Replace with the source dataset's identifier
dataset = load_dataset(source_dataset_name)

# Step 3: Create a new repository under your account
new_dataset_name = "english_emotions"  # Replace with your desired dataset name
username = "your_username_here"  # Replace with your Hugging Face username
repo_id = f"{username}/{new_dataset_name}"

api = HfApi()
create_repo(repo_id=repo_id, token=hf_token, repo_type="dataset")

# Step 4: Save the dataset locally in a format ready for uploading
save_path = "./cloned_dataset"
dataset.save_to_disk(save_path)

# Step 5: Push the dataset to your Hugging Face repository
from huggingface_hub import Repository

repo = Repository(local_dir=save_path, clone_from=repo_id)
repo.git_add(".")
repo.git_commit("Initial commit of cloned dataset")
repo.git_push()

print(f"Dataset cloned and uploaded to: https://huggingface.co/{repo_id}")


###Clone Dataset with conditional row

In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi, HfFolder, create_repo, upload_folder
import os
import shutil

# Step 1: Set up your Hugging Face credentials
hf_token = "YOUR_HF_TOKEN_HERE"  # Replace with your actual token
HfFolder.save_token(hf_token)

# Step 2: Load the dataset you want to clone
source_dataset_name = "ylacombe/expresso-tagged"  # Replace with the source dataset's identifier
dataset = load_dataset(source_dataset_name)

# Step 3: Filter the dataset to keep rows where 'speaker_id' equals 'ex03'
filtered_dataset = dataset.filter(lambda example: example['speaker_id'] == 'Thomas')

# Step 4: Create a new repository under your account
new_dataset_name = "english_emotions_tagged"  # Replace with your desired dataset name
username = "En1gma02"  # Replace with your Hugging Face username
repo_id = f"{username}/{new_dataset_name}"

api = HfApi()
create_repo(repo_id=repo_id, token=hf_token, repo_type="dataset", exist_ok=True)

# Step 5: Save the filtered dataset locally
save_path = "./filtered_cloned_dataset"
if os.path.exists(save_path):  # Clean directory if it exists
    shutil.rmtree(save_path)
filtered_dataset.save_to_disk(save_path)

# Step 6: Upload the dataset to the Hugging Face repository
upload_folder(
    folder_path=save_path,
    repo_id=repo_id,
    repo_type="dataset",
    token=hf_token,
    commit_message="Initial commit of filtered dataset with speaker_id='ex03'"
)

print(f"Filtered dataset cloned and uploaded to: https://huggingface.co/{repo_id}")


README.md:   0%|          | 0.00/487 [00:00<?, ?B/s]

train-00000-of-00012.parquet:   0%|          | 0.00/473M [00:00<?, ?B/s]

train-00001-of-00012.parquet:   0%|          | 0.00/582M [00:00<?, ?B/s]

train-00002-of-00012.parquet:   0%|          | 0.00/505M [00:00<?, ?B/s]

train-00003-of-00012.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

...

data-00001-of-00003.arrow:   0%|          | 0.00/499M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

data-00000-of-00003.arrow:   0%|          | 0.00/411M [00:00<?, ?B/s]

data-00002-of-00003.arrow:   0%|          | 0.00/453M [00:00<?, ?B/s]

Filtered dataset cloned and uploaded to: https://huggingface.co/En1gma02/english_emotions_tagged
